In [78]:
import json 
import pandas as pd 
pd.options.mode.chained_assignment = None

In [79]:
with open("results/ideology_news_dichotomized/regression_results.json") as file:
    res = pd.DataFrame(json.load(file)).T

res["reg-ID"] = [f"R-{i:06}" for i in range(len(res))]

print(f"Out of {len(res)} regressions, {100 * ('FAILED' != res['Coef']).mean():.0f}% were significant")
res_success = res.loc['FAILED' != res['Coef']]
print(f"out of {len(res_success)} successful regressions {100 * (res_success['LLR p-value'] <= 0.05).mean():.0f}% regressions are significative")
res_significative = res_success.loc[res_success["LLR p-value"] <= 0.05]
res_significative.loc[:,"bias"] = res_significative["x_column"].apply(lambda x : str(x).split("-")[0])
valid_for_comparison = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"]).size().sort_values() == 2

print(f"out of {len(res_significative)} significant regressions, in only {100 * valid_for_comparison.mean():.0f} cases, we can compare the regression with the gold standard and predicted labels ({100 * valid_for_comparison.sum() / len(res):.0f}% of all regressions)")

def valid_ids(x: list):
    if len(x) == 2:
        return list(x)
valid_reg_ID = res_significative.groupby(["bias", "y_column", "model", "learning_rate", "weight_decay"], as_index=False)["reg-ID"].aggregate(valid_ids)["reg-ID"].to_list()

Out of 25488 regressions, 95% were significant
out of 24160 successful regressions 28% regressions are significative
out of 6806 significant regressions, in only 52 cases, we can compare the regression with the gold standard and predicted labels (9% of all regressions)


In [80]:
IDS_for_errors_type_1 = []
IDS_for_errors_type_2 = []
IDS_for_errors_type_S = []
IDS_for_errors_type_M = []

N_configurations = sum([int(reg_IDs is not None) for reg_IDs in valid_reg_ID])

for reg_IDs in valid_reg_ID:
    if not reg_IDs:
        continue
    reg_cases = [
        res.loc[res["reg-ID"] == reg_IDs[0],:],
        res.loc[res["reg-ID"] == reg_IDs[1],:]
    ]
    if reg_cases[0]["x_column"].item().endswith("-GS"):
        id_GS, id_pred = 0, 1
    else:
        id_GS, id_pred = 1, 0
        
    GS_significant = reg_cases[id_GS]["pvalues"].item()[1] <= 0.05
    pred_significant = reg_cases[id_pred]["pvalues"].item()[1] <= 0.05

    GS_coef = reg_cases[id_GS]["Coef"].item()[1]
    pred_coef = reg_cases[id_pred]["Coef"].item()[1]

    error_type_1 = pred_significant and not GS_significant
    error_type_2 = GS_significant and not pred_significant
    error_type_S = pred_significant and GS_significant and (GS_coef * pred_coef < 0)

    if error_type_1: IDS_for_errors_type_1 += [reg_IDs]
    if error_type_2: IDS_for_errors_type_2 += [reg_IDs]
    if error_type_S: IDS_for_errors_type_S += [reg_IDs]

print(f"IDS_for_errors_type_1: {len(IDS_for_errors_type_1)} / {N_configurations} ({100 * len(IDS_for_errors_type_1) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_2: {len(IDS_for_errors_type_2)} / {N_configurations} ({100 * len(IDS_for_errors_type_2) / N_configurations:.0f}%)")
print(f"IDS_for_errors_type_S: {len(IDS_for_errors_type_S)} / {N_configurations} ({100 * len(IDS_for_errors_type_S) / N_configurations:.0f}%)")

IDS_for_errors_type_1: 1178 / 2320 (51%)
IDS_for_errors_type_2: 9 / 2320 (0%)
IDS_for_errors_type_S: 2 / 2320 (0%)
